# Playing with incremental ingestion & processing

In the previous notebook, we only ingested the data for the same interval 2025-01-01
00:00 until 03:00. This notebook continues on that, focusing on how incremental
processing works in SQLMesh. This is more confusing than I initially expected.

**Note:** as before, you can use the `make clean` command to wipe the state and data
databases completely to start over. SQLMesh does not have such a command built in (and
that's a good thing - it would be quite dangerous with a real data warehouse).

Let's start by ingesting the data of 00:00-03:00 again in both dev and acc:

In [ ]:
!cd /workspaces/sqlmesh_playground/ && sqlmesh plan dev --execution-time '2025-01-01 03:00' --auto-apply

In [ ]:
!cd /workspaces/sqlmesh_playground/ && sqlmesh plan acc --execution-time '2025-01-01 03:00' --auto-apply

Similarly as in the previous exercise, the run in `acc` should only be a virtual
update because it can reuse the data in the physical layer from the run in dev.

Let's now ingest the data from 03:00-04:00 in dev too:

In [ ]:
!cd /workspaces/sqlmesh_playground/ && sqlmesh plan dev --execution-time '2025-01-01 04:00' --auto-apply

Note how this neatly shows which data intervals will be ingested and transformed,
and which models will be fully refreshed:

```log
  Models needing backfill:
  ├── gharchive__dev.commits: [2025-01-01 03:00:00 - 2025-01-01 03:59:59]
  ├── gharchive__dev.commits_stats_per_hour: [2025-01-01 03:00:00 - 2025-01-01 
  │   03:59:59]
  ├── gharchive__dev.commits_stats_total: [full refresh]
  (etc)
```

Let's query the model `commits_stats_per_hour` to verify that we have indeed ingested
and processed the new batch of data:

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT *
    FROM persistent.gharchive__dev.commits_stats_per_hour
    ORDER BY hour ASC
    """
)

Let's query the same model **in environment acc** to see what data is in
`commits_stats_per_hour` there:

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT *
    FROM persistent.gharchive__acc.commits_stats_per_hour
    ORDER BY hour ASC
    """
)

😯

Surprisingly (to me at least), in acc, the data from 03:00-04:00 is also ingested even
though we didn't do a plan or a run in acc (only in dev).

The reason is that for incremental models, there is **only one underlying table** in the
physical layer, not one per batch, which we can double check by looking at the model's
view definition:

In [ ]:
import notebook_utils
notebook_utils.query_duckdb(
    """
    SELECT table_schema, table_name, view_definition
    FROM information_schema.views
    WHERE table_schema like 'gharchive__%'
    AND table_name = 'funny_commits'
    """
)

They just point to the same underlying table (gharchive__funny_commits__1489903899),
without any filters on time ranges.

So any change to that table in any environment also affects all other environments,
as long as the code of the model (and all upstream models) remains unchanged.
This is a surprising gotcha to be aware of when using SQLMesh.

It makes sense though, because if each data interval would be its own physical table,
then that might have significant performance impact if you have many such data intervals.

Of course, if the code of the model (or an upstream model) changes, then a different
table would be used in the physical layer in that environment, similarly as with
non-incremental models.